<a href="https://colab.research.google.com/github/jesusessu/MDD_LAB16/blob/develop/MDD_LAB16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SEMANA 16: DESARROLLO DE APLICACIONES DE MACHINE LEARNING**

Esplana Sulla Jesús Zósimo

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Descargar datos desde UCI
!wget -O adult.data https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
!wget -O adult.test https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test

# Nombres de columnas
columnas = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

# Cargar datos
df_train = pd.read_csv('adult.data', names=columnas, na_values=' ?', skipinitialspace=True)
df_test = pd.read_csv('adult.test', names=columnas, skiprows=1, na_values=' ?', skipinitialspace=True)

# Unir
df = pd.concat([df_train, df_test], ignore_index=True)

# Limpiar etiquetas
df['income'] = df['income'].str.replace('.', '', regex=False)
df.head()

--2025-07-06 00:35:14--  https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘adult.data’

adult.data              [   <=>              ]   3.79M  7.79MB/s    in 0.5s    

2025-07-06 00:35:15 (7.79 MB/s) - ‘adult.data’ saved [3974305]

--2025-07-06 00:35:15--  https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘adult.test’

adult.test              [   <=>              ]   1.91M  4.22MB/s    in 0.5s    

2025-07-06 00:35:16 (4.22 MB/s) - ‘adult.test’ saved [2003153]



,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
# Verificar NA
print("Valores nulos:\n", df.isnull().sum())

# Imputación simple para valores nulos
for col in df.select_dtypes(include='object'):
    df[col] = df[col].fillna(df[col].mode()[0])

# Outliers univariados
for col in ['age', 'hours-per-week', 'capital-gain', 'capital-loss']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lim_sup = q3 + 1.5 * iqr
    lim_inf = q1 - 1.5 * iqr
    df[col] = np.where(df[col] > lim_sup, lim_sup, df[col])
    df[col] = np.where(df[col] < lim_inf, lim_inf, df[col])

Valores nulos:
 age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


In [6]:
# Dummies
X = df.drop('income', axis=1)
y = df['income'].map({'<=50K': 0, '>50K': 1})

X_encoded = pd.get_dummies(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

# División
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Balanceo
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

In [7]:
models = {
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'LogReg': LogisticRegression(),
    'Tree': DecisionTreeClassifier(),
    'RF': RandomForestClassifier()
}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    preds = model.predict(X_test)
    print(f"\n{name} Accuracy: {accuracy_score(y_test, preds):.4f}")


KNN Accuracy: 0.7665

SVM Accuracy: 0.7792

LogReg Accuracy: 0.7885

Tree Accuracy: 0.7854

RF Accuracy: 0.8156


In [8]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3, scoring='accuracy')
grid.fit(X_train_bal, y_train_bal)

print("Mejor modelo:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, grid.predict(X_test)))

Mejor modelo: {'max_depth': None, 'n_estimators': 200}
Accuracy: 0.8162555020984748
